BI-DIRECTIONAL GRU CLASSIFIER USING LOG RETURNS

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(42)

# 1. Load and Clean
df = pd.read_csv('NIFTY50.csv')
df = df.dropna(axis=1, how='all')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df = df.dropna()

# 2. Stationary Features (Log Returns)
df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))
df['Vol_Log'] = np.log(df['Volume'] + 1)
df['High_Low_Pct'] = (df['High'] - df['Low']) / df['Low']
df['Close_Open_Pct'] = (df['Close'] - df['Open']) / df['Open']

# Target
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.dropna().reset_index(drop=True)

# Feature Selection
feature_cols = ['Log_Ret', 'Vol_Log', 'High_Low_Pct', 'Close_Open_Pct']
X_data = df[feature_cols].values
y_data = df['Target'].values

# 3. Chronological Split
train_size = int(len(df) * 0.8)
X_train_raw, X_test_raw = X_data[:train_size], X_data[train_size:]
y_train_raw, y_test_raw = y_data[:train_size], y_data[train_size:]

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 4. Create Sequences
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 30 # Increased lookback
X_train, y_train = create_sequences(X_train_scaled, y_train_raw, time_steps)
X_test, y_test = create_sequences(X_test_scaled, y_test_raw, time_steps)

# 5. Class Weights (to fix long bias)
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))
print(f"Class Weights: {class_weights}")

# 6. Bidirectional GRU Model
model = Sequential([
    Bidirectional(GRU(64, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Bidirectional(GRU(32, return_sequences=False)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# 7. Train
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    class_weight=class_weights,
    verbose=0
)

# 8. Evaluation & Report
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

report = classification_report(y_test, y_pred, target_names=['Down/Same', 'Up'], output_dict=True)
report_df = pd.DataFrame(report).transpose()
print(classification_report(y_test, y_pred, target_names=['Down/Same', 'Up']))


Class Weights: {0: np.float64(1.0583279325988335), 1: np.float64(0.9477655252466628)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step
              precision    recall  f1-score   support

   Down/Same       0.43      0.17      0.24       356
          Up       0.55      0.82      0.66       439

    accuracy                           0.53       795
   macro avg       0.49      0.49      0.45       795
weighted avg       0.49      0.53      0.47       795

